# ATML PA2 - Kaggle runner

**Settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On** (needs a phone-verified Kaggle account).

**How to run:** set `REPO_URL` below, then **Save Version -> Save & Run All (Commit)**. It runs in the background
(max 12 h; the full run is ~6-7 h on a T4) and everything in `/kaggle/working` (adapters, results,
figures, logs, `pa2_results.tgz`) is kept as this version's Output.

**If a run stops early:** open the notebook, *Add Input -> Notebook Output ->* pick the previous version,
set `RESUME_FROM` to its path (e.g. `/kaggle/input/atml-pa2-runner`), and commit again. Finished stages are skipped.

In [ ]:
REPO_URL = "https://github.com/laeebahafeez/ATML-PA2.git"   # <-- your GitHub repo
BRANCH = "main"
STAGES = "t1 t2 t3 t4 t5 fig"   # subset e.g. "t4 t5 fig" if you only need those
RESUME_FROM = ""                 # e.g. "/kaggle/input/atml-pa2-runner" (a previous version's output)

In [ ]:
import os, subprocess, glob
WORK = '/kaggle/working/pa2'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, WORK], check=True)
os.chdir(WORK)
!git log --oneline -1
if RESUME_FROM:
    # previous output may hold pa2/ directly or only the tarball
    src = os.path.join(RESUME_FROM, 'pa2')
    if os.path.isdir(src):
        for d in ['outputs', 'results']:
            if os.path.isdir(os.path.join(src, d)):
                !cp -rn {src}/{d} {WORK}/
    for tgz in glob.glob(os.path.join(RESUME_FROM, '*.tgz')):
        !tar xzf {tgz} -C {WORK} --skip-old-files
    !ls outputs/* results 2>/dev/null | head -50
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install -r requirements.txt
!python -m scripts.download_assets
!python -m scripts.validate_assets
!python -m scripts.check_environment
!python -m pytest tests/test_objectives.py -q

In [ ]:
# Main run. Each stage is resumable; a failure stops the script but later cells still save what finished.
!bash scripts/run_all.sh {STAGES} 2>&1 | tee -a results/run_log.txt

In [ ]:
# Package small results (LoRA adapters are ~4 MB each, no base weights) for download.
!python -m report.make_figures
!tar czf /kaggle/working/pa2_results.tgz outputs results report/figures report/tables.md
!du -sh /kaggle/working/pa2_results.tgz
!tail -40 results/run_log.txt